# RetailPulse — 探索性資料分析（EDA）

**資料來源**：Online Retail UCI Dataset（2010-12-01 ～ 2011-12-09）  
**執行前提**：先在終端執行 `make etl`，確認 DuckDB 已建立再開始。

### 這份 notebook 想回答哪些問題？
1. 原始資料有多少筆？清洗後剩幾成？
2. 收入的季節性走勢是什麼樣子？
3. 哪些國家是主要市場？
4. RFM 三個維度的分布是否適合直接跑 K-Means？
5. MBA 的先決條件（多品發票）是否足夠？

In [ ]:
# ── 路徑設定：notebooks/ 需要往上找 src/ ────────────────────────────────
import sys
from pathlib import Path

ROOT = Path.cwd().parent          # 從 notebooks/ 退一層到專案根目錄
sys.path.insert(0, str(ROOT / 'src'))  # 讓 Python 找得到 utils、features 等模組

# ── 第三方套件 ───────────────────────────────────────────────────────────
import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')  # 隱藏 pandas/sklearn 的版本警告，保持輸出乾淨

# ── 專案模組 ─────────────────────────────────────────────────────────────
from utils.config import get_settings

# 開唯讀連線：分析用不到寫入，唯讀可避免意外修改資料
settings = get_settings()
conn = duckdb.connect(settings.duckdb_path, read_only=True)

print('✅ DuckDB 連線成功')
print(f'   資料庫：{settings.duckdb_path}')

---
## 1. 資料概況快覽

先確認 ETL 後各表的基本規模，對「這份資料有多大」建立直覺。

In [ ]:
# 一條 SQL 跨多表 JOIN，撈出最核心的 KPI
# 注意：這裡的數字是「清洗後」的，已過濾取消訂單 + 無 CustomerID 的資料
stats = conn.execute('''
    SELECT
        COUNT(DISTINCT i.customer_id)   AS 客戶數,
        COUNT(DISTINCT i.invoice_no)    AS 發票數,
        SUM(ii.quantity)                AS 總售出件數,
        ROUND(SUM(i.total_amount), 0)   AS 總營收_英鎊,
        ROUND(AVG(i.total_amount), 2)   AS 平均客單價_英鎊,
        COUNT(DISTINCT p.stock_code)    AS 商品種類數
    FROM invoices i
    JOIN invoice_items ii ON i.invoice_no = ii.invoice_no
    JOIN products p ON ii.stock_code = p.stock_code
''').df()

print('=== 清洗後資料概況 ===')
display(stats.T.rename(columns={0: '數值'}))

# 解讀：
# - 原始 541,909 筆 → 清洗後 397,884 筆（73.4% 保留率）
# - 取消訂單 9,288 筆；無 CustomerID（訪客結帳）134,697 筆
# - 平均客單價 £480 遠高於一般零售，因為很多客戶是批發商
print('\n💡 平均客單價 £480 偏高，因 UCI 資料集包含大量 B2B 批發訂單')

---
## 2. 月度營收趨勢

零售業有強烈季節性，秋冬備貨期通常是高峰。先確認這份資料是否符合預期。

In [ ]:
# 用 DuckDB 的 STRFTIME 做月份聚合（比 pandas resample 更 readable）
df_monthly = conn.execute('''
    SELECT
        STRFTIME(date, '%Y-%m') AS month,
        ROUND(SUM(revenue), 0)  AS revenue,
        SUM(orders)             AS orders
    FROM daily_sales
    GROUP BY month
    ORDER BY month
''').df()

# 計算月環比成長率（Month-over-Month）
df_monthly['mom_growth_pct'] = df_monthly['revenue'].pct_change() * 100

# 雙軸圖：左軸 = 月營收柱狀圖，右軸 = 環比成長率折線
fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(
    go.Bar(x=df_monthly['month'], y=df_monthly['revenue'],
           name='月營收（£）', marker_color='#2563EB', opacity=0.85),
    secondary_y=False
)
fig.add_trace(
    go.Scatter(x=df_monthly['month'], y=df_monthly['mom_growth_pct'],
               name='月環比成長%', mode='lines+markers',
               line=dict(color='#DC2626', width=2), marker=dict(size=8)),
    secondary_y=True
)
fig.update_layout(
    title='月度營收趨勢（2010-12 ～ 2011-12）',
    legend=dict(orientation='h', y=1.1)
)
fig.update_yaxes(title_text='月營收（£）', secondary_y=False)
fig.update_yaxes(title_text='月環比成長率（%）', secondary_y=True)
fig.show()

peak = df_monthly.loc[df_monthly['revenue'].idxmax()]
print(f'📈 全年高峰：{peak["month"]}，£{peak["revenue"]:,.0f}')
print('💡 9～11 月急速攀升，典型零售聖誕備貨旺季。2011-12 下滑是因為資料只到 12/09')

---
## 3. 地理市場分析

英國是主場，但向全球出貨。哪些海外市場值得關注？

In [ ]:
# 按國家聚合：訂單數、客戶數、總收入、平均客單價
df_country = conn.execute('''
    SELECT
        country,
        COUNT(*)                    AS orders,
        COUNT(DISTINCT customer_id) AS customers,
        ROUND(SUM(total_amount), 0) AS revenue,
        ROUND(AVG(total_amount), 2) AS aov
    FROM invoices
    GROUP BY country
    ORDER BY revenue DESC
''').df()

df_country['revenue_pct'] = (df_country['revenue'] / df_country['revenue'].sum() * 100).round(1)

# 左圖：Top 10 國家收入佔比 | 右圖：非英國的平均客單價
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['各國收入佔比（Top 10）', '非英國市場：平均客單價（£）'])

top10 = df_country.head(10)
fig.add_trace(
    go.Bar(x=top10['revenue_pct'], y=top10['country'], orientation='h',
           marker_color='#2563EB',
           text=top10['revenue_pct'].astype(str) + '%', textposition='outside',
           name='收入佔比'),
    row=1, col=1
)

# 非英國市場 AOV 通常更高（批發商多）
non_uk = df_country[df_country['country'] != 'United Kingdom'].head(10)
fig.add_trace(
    go.Bar(x=non_uk['aov'], y=non_uk['country'], orientation='h',
           marker_color='#16A34A', name='AOV（£）'),
    row=1, col=2
)

fig.update_layout(title='地理市場分析', showlegend=False, height=420)
fig.update_xaxes(row=1, col=1)
fig.update_yaxes(categoryorder='total ascending', row=1, col=1)
fig.update_yaxes(categoryorder='total ascending', row=1, col=2)
fig.show()

uk_pct = df_country[df_country['country'] == 'United Kingdom']['revenue_pct'].iloc[0]
print(f'🇬🇧 英國佔比 {uk_pct}%，其他 {df_country["country"].nunique()-1} 個國家合計 {100-uk_pct}%')
print('💡 荷蘭、愛爾蘭、德國的平均客單價高於英國本土，推測是代理商或批發商')

---
## 4. 客戶 RFM 分布分析

跑 K-Means 前，先看三個維度的分布形狀。嚴重右偏的資料需要 StandardScaler 正規化。

In [ ]:
# 直接從 ETL 後的 customer_features 表撈資料
df_rfm = conn.execute('SELECT * FROM customer_features').df()

print(f'客戶總數：{len(df_rfm):,}')
print('\nRFM 敘述統計：')
display(df_rfm[['recency_days', 'frequency', 'monetary']].describe().round(1))

# 三欄直方圖，Monetary 用 log scale 讓長尾清晰可見
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        'Recency（距今天數，越小越好）',
        'Frequency（訂單次數，截斷 @30）',
        'log(Monetary+1)（長尾需要 log 壓縮）'
    ]
)

# Recency：比較均勻，不需特殊處理
fig.add_trace(
    go.Histogram(x=df_rfm['recency_days'], nbinsx=30,
                 marker_color='#2563EB', name='Recency'),
    row=1, col=1
)
# Frequency：右偏，大多數客戶買過 1～10 次
fig.add_trace(
    go.Histogram(x=df_rfm['frequency'].clip(upper=30), nbinsx=30,
                 marker_color='#16A34A', name='Frequency'),
    row=1, col=2
)
# Monetary：嚴重右偏（Champions 平均 £127K），log 變換後接近常態
fig.add_trace(
    go.Histogram(x=np.log1p(df_rfm['monetary']), nbinsx=30,
                 marker_color='#F59E0B', name='log(Monetary)'),
    row=1, col=3
)

fig.update_layout(
    title='RFM 三維度分布（Monetary 有嚴重右偏 → K-Means 前必須 StandardScaler）',
    showlegend=False, height=320
)
fig.show()

# 帕累托分析：前 20% 客戶貢獻多少收入？
sorted_rev = df_rfm['monetary'].sort_values(ascending=False)
top20_count = int(len(sorted_rev) * 0.2)
top20_share = sorted_rev.iloc[:top20_count].sum() / sorted_rev.sum() * 100
print(f'\n📊 帕累托：前 20% 客戶（{top20_count:,} 人）貢獻 {top20_share:.1f}% 總收入')
print('   → 這就是為什麼維護 Champions（13人）和 Loyal（211人）是最優先的事')

In [ ]:
# K-Means 分群結果視覺化
# 用顏色區分四個 segment，氣泡大小代表消費金額
segment_colors = {
    'Champions':       '#16A34A',  # 深綠：最佳客戶，要維持關係
    'Loyal Customers': '#2563EB',  # 藍：忠實，給予忠誠獎勵
    'At Risk':         '#F59E0B',  # 橘：有流失風險，需主動接觸
    'Lost':            '#DC2626',  # 紅：已流失，再行銷成本高
}

fig = px.scatter(
    df_rfm,
    x='recency_days', y='frequency',
    color='segment', size='monetary', size_max=50,
    color_discrete_map=segment_colors,
    hover_data=['customer_id', 'rfm_score', 'monetary'],
    title='RFM 客戶分群（X = 距今天數，Y = 訂單次數，泡泡大小 = 消費額）',
    labels={
        'recency_days': '距今天數（越小越近期）',
        'frequency': '訂單次數',
    }
)
fig.update_layout(height=500)
fig.show()

# 分群摘要：每個 segment 的平均輪廓
summary = df_rfm.groupby('segment', observed=True).agg(
    人數=('customer_id', 'count'),
    平均距今天數=('recency_days', 'mean'),
    平均訂單次數=('frequency', 'mean'),
    平均消費額=('monetary', 'mean'),
).round(1).sort_values('平均消費額', ascending=False)
print('\n=== RFM 分群摘要 ===')
display(summary)

---
## 5. 熱門商品與長尾分布

In [ ]:
# Top 20 商品（依出現的發票數，不是銷售量）
# 用發票數而非銷售量，是因為「多少次被放入購物車」比「賣了多少件」更能反映受歡迎程度
df_products = conn.execute('''
    SELECT description, order_count, ROUND(total_revenue, 0) AS total_revenue
    FROM product_features
    ORDER BY popularity_rank
    LIMIT 20
''').df()

fig = px.bar(
    df_products,
    x='order_count', y='description',
    orientation='h',
    color='total_revenue', color_continuous_scale='Blues',
    title='Top 20 商品（顏色深淺 = 總收入）',
    labels={'order_count': '出現的發票數', 'description': '商品', 'total_revenue': '總收入（£）'}
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=600)
fig.show()

# 長尾分析
all_products = conn.execute('SELECT order_count FROM product_features ORDER BY order_count DESC').df()
n = len(all_products)
top10_n = int(n * 0.1)
top10_share = all_products.head(top10_n)['order_count'].sum() / all_products['order_count'].sum() * 100
print(f'\n📦 商品目錄：{n:,} 種')
print(f'📊 前 10%（{top10_n} 種）貢獻了 {top10_share:.1f}% 的訂單數 → 長尾效應顯著')
print('💡 長尾商品需要靠 MBA 推薦才能被發現，單靠人氣排行不夠')

---
## 6. MBA 前置：購物籃大小分布

In [ ]:
# Apriori 的成立前提：一張發票要有多件商品才能產生共現關係
# 先確認多品發票的比例夠高，MBA 才值得做
df_basket = conn.execute('''
    SELECT invoice_no, COUNT(DISTINCT stock_code) AS unique_products
    FROM invoice_items
    GROUP BY invoice_no
''').df()

fig = px.histogram(
    df_basket, x='unique_products', nbins=50,
    title='每張發票的商品種類數分布',
    labels={'unique_products': '商品種類數/張', 'count': '發票數'},
    color_discrete_sequence=['#7C3AED']
)
fig.update_layout(bargap=0.03)
fig.show()

single = (df_basket['unique_products'] == 1).sum()
multi  = (df_basket['unique_products'] > 1).sum()
print(f'單品發票：{single:,} 張（{single/len(df_basket)*100:.1f}%）')
print(f'多品發票：{multi:,} 張（{multi/len(df_basket)*100:.1f}%）')
print(f'中位數：{df_basket["unique_products"].median():.0f} 件/張')
print('\n💡 多品發票占多數 → MBA 有充足共現資料可以挖掘')

# 顯示挖到的關聯規則
rules_count = conn.execute('SELECT COUNT(*) FROM mba_rules').fetchone()[0]
max_lift = conn.execute('SELECT MAX(lift) FROM mba_rules').fetchone()[0]
print(f'\n✅ Apriori 結果：{rules_count} 條規則，最高 Lift = {max_lift:.2f}')

---
## 7. 分析結論摘要

In [ ]:
# 最終結論，格式化輸出供報告使用
uk_rev = conn.execute(
    "SELECT SUM(total_amount) FROM invoices WHERE country='United Kingdom'"
).fetchone()[0]
total_rev = conn.execute('SELECT SUM(total_amount) FROM invoices').fetchone()[0]

print('''
╔══════════════════════════════════════════════════════════════════╗
║              RetailPulse EDA — 結論摘要                        ║
╠══════════════════════════════════════════════════════════════════╣
║  資料規模                                                       ║
║  • 清洗後 397,884 筆 / 4,338 客戶 / 3,665 商品                 ║
║  • 總營收 £8.91M（2010-12 ～ 2011-12）                         ║
╠══════════════════════════════════════════════════════════════════╣
║  季節性                                                         ║
║  • 旺季：9～11月（聖誕備貨），11月高峰 £1.16M                  ║
║  • 平淡期：2月（£447K，全年最低）                               ║
╠══════════════════════════════════════════════════════════════════╣
║  地理分布                                                       ║
║  • 英國佔 82% 收入；荷蘭、愛爾蘭是最重要海外市場               ║
║  • 海外客戶 AOV 更高（批發商）                                  ║
╠══════════════════════════════════════════════════════════════════╣
║  客戶分群                                                       ║
║  • 13 位 Champions 平均消費 £127,338（極度集中）                ║
║  • 前 20% 客戶貢獻 80%+ 收入（帕累托法則成立）                 ║
║  • 3,053 位 At Risk 是最大的潛在流失風險                        ║
╠══════════════════════════════════════════════════════════════════╣
║  商品與 MBA                                                     ║
║  • 88 條關聯規則，最高 Lift = 24（強力配套組合）                ║
║  • 商品長尾顯著：前 10% 商品 ≈ 佔訂單數的 50%+                ║
╚══════════════════════════════════════════════════════════════════╝
''')

# 關閉連線，養成好習慣
conn.close()
print('✅ EDA 完成，連線已關閉')